In [ ]:
import yfinance as yf
import pandas as pd

# COAL & GAS
TICKERS = {
    "coal_USD_t":  "MTF=F",   # Rotterdam Coal Futures [USD/ton]
    "gas_EUR_MWh": "TTF=F",   # TTF Natural Gas Futures [EUR/MWh] — benchmark alemán
    "EURUSD":      "USDEUR=X", # Tipo de cambio para convertir coal a EUR
}

raw = yf.download(
    list(TICKERS.values()),
    start="2021-12-29",
    end="2026-01-03",
    interval="1d",
    auto_adjust=True,
    progress=False
)["Close"]

# Renombrar columnas
reverse = {v: k for k, v in TICKERS.items()}
raw = raw.rename(columns=reverse)

# Convertir coal de USD/ton → EUR/ton
raw["coal_EUR_t"] = raw["coal_USD_t"]*raw["EURUSD"]

# Limpiar
commodities = raw[["coal_USD_t", "coal_EUR_t", "gas_EUR_MWh"]].copy()
commodities.index = pd.to_datetime(commodities.index).tz_localize("Europe/Berlin", 
                                                                    ambiguous="NaT", 
                                                                    nonexistent="NaT")
commodities.index.name = "timestamp"

# Rellenar fines de semana/festivos con forward fill (mercados cerrados)
commodities = commodities.resample("h").ffill()  # expandir a horario como el resto del dataset

print(f"  Shape: {commodities.shape}")
print(f"  Nulos: {commodities.isna().sum().to_dict()}")
print(commodities.tail())

# Rellenar fines de semana/festivos con forward fill (mercados cerrados)
commodities = commodities.resample("h").ffill()

# ← AÑADIR ESTO
commodities = commodities[commodities.index >= "2022-01-01"]
commodities = commodities[commodities.index <= "2026-01-01"]

print(f"  Shape: {commodities.shape}")

commodities.to_csv("COMMODITIES_2025.csv")

In [ ]:
commodities